<a href="https://colab.research.google.com/github/sataraw/se3-grasp-flow/blob/gsfatakhov%2FM2/M2/notebooks/m2_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M2: MuJoCo grasp evaluation on Colab

Runs `scripts/eval_mujoco.py` on a Colab machine: calibration with ground-truth and random grasps, then
EquiGraspFlow across sampling budgets (NFE).

**Before the first run**

1. In Google Drive, the two shared folders linked in the repository README (ACRONYM dataset and
   EquiGraspFlow model) must be reachable from **My Drive**: either as a shortcut
   (**Organise → Add shortcut → My Drive**) or as a copy with the same folder structure.
2. Put their paths into the settings cell below. The defaults are the names of the shared folders,
   `dataset` and `pretrained_models`.
3. **Runtime → Change runtime type → GPU** (needed for the latency numbers; the simulation itself runs on CPU).
4. The branch named in the settings cell must be pushed to GitHub.

Everything runs through `uv run`, in the project's own Python 3.10 environment, not in the notebook
kernel. Results and the convex-decomposition cache are copied to Drive by the last cell; `/content` is
wiped when the session ends.

In [3]:
# Settings
REPO = "https://github.com/sataraw/se3-grasp-flow.git"
BRANCH = "gsfatakhov/M2"

# The shared folders as they appear in "My Drive" (shortcut or copy)
DRIVE_DATASET = "/content/drive/MyDrive/dataset"          # contains grasps/ meshes/ splits/
DRIVE_CHECKPOINTS = "/content/drive/MyDrive/pretrained_models"  # contains equigraspflow_full/ ...

# Where results and the decomposition cache are kept between sessions
DRIVE_OUT = "/content/drive/MyDrive/se3-grasp-flow"

In [4]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Code and environment

In [5]:
import os

%cd /content
if not os.path.exists("se3-grasp-flow"):
    !git clone --recurse-submodules -b {BRANCH} {REPO}
%cd /content/se3-grasp-flow
!git pull --ff-only && git submodule update --init
!pip install -q uv
!uv sync

/content
/content/se3-grasp-flow
Already up to date.
Resolved 117 packages in 1ms
Checked 115 packages in 1ms


## Data

Copied to the local disk, because Drive is slow for thousands of small files. A cache from an earlier
session is restored if there is one.

In [6]:
from pathlib import Path

for path, expected in [(DRIVE_DATASET, ["grasps", "meshes", "splits"]), (DRIVE_CHECKPOINTS, ["equigraspflow_full"])]:
    missing = [name for name in expected if not (Path(path) / name).exists()]
    assert not missing, f"{path}: {missing} not found. Check the shortcut name in the settings cell."

!mkdir -p data/dataset data/checkpoints data/cache
!rsync -a --info=progress2 "{DRIVE_DATASET}/" data/dataset/
!rsync -a --info=progress2 "{DRIVE_CHECKPOINTS}/" data/checkpoints/
if os.path.exists(f"{DRIVE_OUT}/cache"):
    !rsync -a "{DRIVE_OUT}/cache/" data/cache/
!du -sh data/*

              0   0%    0.00kB/s    0:00:00 (xfr#0, to-chk=0/813)
              0   0%    0.00kB/s    0:00:00 (xfr#0, to-chk=0/7)
3.3M	data/cache
15M	data/checkpoints
714M	data/dataset


## Checks

All tests should pass; `test_cuda_matmul` must not be skipped on a GPU runtime.

In [7]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc
!uv run pytest -q -rs

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
12
......................................                                   [100%]
38 passed in 39.04s


## Smoke run

Two objects per category, 20 grasps each. Ground-truth grasps should mostly succeed and random grasps
fail. If ground truth mostly *collides*, the grasp frame convention is wrong
(`T_HAND_FROM_GRASP` in `src/graspgen/sim/gripper.py`).

In [8]:
!uv run python scripts/eval_mujoco.py --source gt random --max-objects 2 --num-grasps 20

24 test cases x 20 grasps, 11 workers

gt: /content/se3-grasp-flow/results/mujoco/calibration/gt
{
  "success_rate": {
    "mean": 0.9895833333333334,
    "per_category": {
      "Laptop": 1.0,
      "Mug": 0.975,
      "Bowl": 0.9916666666666667,
      "Pencil": 0.9916666666666667
    }
  },
  "collision_rate": {
    "mean": 0.0,
    "per_category": {
      "Laptop": 0.0,
      "Mug": 0.0,
      "Bowl": 0.0,
      "Pencil": 0.0
    }
  },
  "contact_rate": {
    "mean": 1.0,
    "per_category": {
      "Laptop": 1.0,
      "Mug": 1.0,
      "Bowl": 1.0,
      "Pencil": 1.0
    }
  }
}

random: /content/se3-grasp-flow/results/mujoco/calibration/random
{
  "success_rate": {
    "mean": 0.0020833333333333333,
    "per_category": {
      "Laptop": 0.008333333333333333,
      "Mug": 0.0,
      "Bowl": 0.0,
      "Pencil": 0.0
    }
  },
  "collision_rate": {
    "mean": 0.16666666666666666,
    "per_category": {
      "Laptop": 0.24166666666666667,
      "Mug": 0.13333333333333333,
      "

## Calibration on the full test set

Targets: ground truth ≳ 90 % success, random ≈ 0 %. The first run decomposes all test meshes, which takes
a while. If ground truth falls short, look at `results/mujoco/calibration/gt/per_case.csv` and adjust
`SIM_ARGS` (e.g. `--coacd-threshold 0.03`, `--collision-tol 0.002`, `--grip-force 60`), then rerun.

In [9]:
SIM_ARGS = ""  # simulation settings, used for every run below

!uv run python scripts/eval_mujoco.py --source gt random {SIM_ARGS}

249 test cases x 100 grasps, 11 workers

gt: /content/se3-grasp-flow/results/mujoco/calibration/gt
{
  "success_rate": {
    "mean": 0.9888755020080321,
    "per_category": {
      "Laptop": 0.9935238095238095,
      "Mug": 0.9754545454545454,
      "Bowl": 0.9933333333333334,
      "Pencil": 0.9943749999999999
    }
  },
  "collision_rate": {
    "mean": 0.0,
    "per_category": {
      "Laptop": 0.0,
      "Mug": 0.0,
      "Bowl": 0.0,
      "Pencil": 0.0
    }
  },
  "contact_rate": {
    "mean": 1.0,
    "per_category": {
      "Laptop": 1.0,
      "Mug": 1.0,
      "Bowl": 1.0,
      "Pencil": 1.0
    }
  }
}

random: /content/se3-grasp-flow/results/mujoco/calibration/random
{
  "success_rate": {
    "mean": 0.0013654618473895584,
    "per_category": {
      "Laptop": 0.002,
      "Mug": 0.00045454545454545455,
      "Bowl": 0.0016666666666666668,
      "Pencil": 0.0010416666666666667
    }
  },
  "collision_rate": {
    "mean": 0.2209236947791165,
    "per_category": {
      "La

In [10]:
import pandas as pd

per_case = pd.read_csv("results/mujoco/calibration/gt/per_case.csv")
display(per_case.groupby("category")[["success_rate", "collision_rate", "contact_rate"]].mean().round(3))
display(per_case.sort_values("success_rate").head(10))

,success_rate,collision_rate,contact_rate
category,,,
Bowl,0.993,0.0,1.0
Laptop,0.994,0.0,1.0
Mug,0.975,0.0,1.0
Pencil,0.994,0.0,1.0


,category,obj_idx,rot_idx,name,num_parts,mass,success_rate,collision_rate,contact_rate
88,Laptop,126,1,Laptop_bdffc02ab69552091185c4b7285136bf_0.0018...,9,0.417746,0.76,0.0,1.0
100,Laptop,32,1,Laptop_2a457d1b101705c445bdacf7bcfe36bf_0.0239...,5,0.366693,0.91,0.0,1.0
148,Mug,66,1,Mug_b7841572364fd9ce1249ffc39a0c3c0b_0.0005132...,25,0.084421,0.92,0.0,1.0
145,Mug,65,1,Mug_b7841572364fd9ce1249ffc39a0c3c0b_0.0003944...,25,0.038318,0.93,0.0,1.0
167,Mug,33,2,Mug_542235fc88d22e1e3406473757712946_0.0012365...,26,0.054148,0.93,0.0,1.0
157,Mug,94,1,Mug_e984fd7e97c2be347eaeab1f0c9120b7_2.9440685...,53,0.015737,0.94,0.0,1.0
89,Laptop,126,2,Laptop_bdffc02ab69552091185c4b7285136bf_0.0018...,9,0.417746,0.94,0.0,1.0
150,Mug,13,0,Mug_23fb2a2231263e261a9ac99425d3b306_0.0004178...,75,0.013948,0.94,0.0,1.0
239,Pencil,62,2,Pencil_a7787ba523465ea1ef2e227a41551d79_0.0004...,1,0.009629,0.94,0.0,1.0
166,Mug,33,1,Mug_542235fc88d22e1e3406473757712946_0.0012365...,26,0.054148,0.95,0.0,1.0


## EquiGraspFlow across NFE

Euler sampler at NFE 1, 2, 5, 10, 20, then the paper's default sampler (RK4-MK, 20 steps = NFE 80).

In [11]:
!uv run python scripts/eval_mujoco.py --solver SE3_Euler --steps 1 2 5 10 20 {SIM_ARGS}
!uv run python scripts/eval_mujoco.py {SIM_ARGS}

249 test cases x 100 grasps, 11 workers

SE3_Euler-1: /content/se3-grasp-flow/results/mujoco/equigraspflow_full/SE3_Euler-1
{
  "success_rate": {
    "mean": 0.006506024096385543,
    "per_category": {
      "Laptop": 0.00038095238095238096,
      "Mug": 0.02303030303030303,
      "Bowl": 0.002,
      "Pencil": 0.0
    }
  },
  "collision_rate": {
    "mean": 0.5602811244979921,
    "per_category": {
      "Laptop": 0.46847619047619043,
      "Mug": 0.6292424242424243,
      "Bowl": 0.027000000000000003,
      "Pencil": 0.9995833333333333
    }
  },
  "contact_rate": {
    "mean": 0.20164658634538152,
    "per_category": {
      "Laptop": 0.2887619047619048,
      "Mug": 0.21636363636363634,
      "Bowl": 0.018666666666666668,
      "Pencil": 0.10520833333333333
    }
  }
}

SE3_Euler-2: /content/se3-grasp-flow/results/mujoco/equigraspflow_full/SE3_Euler-2
{
  "success_rate": {
    "mean": 0.11144578313253013,
    "per_category": {
      "Laptop": 0.13495238095238096,
      "Mug": 0.07

In [12]:
import json
from pathlib import Path

rows = []
for path in sorted(Path("results/mujoco").glob("*/*/summary.json")):
    s = json.loads(path.read_text())
    rows.append({
        "run": path.parent.parent.name,
        "name": path.parent.name,
        "nfe": s.get("nfe"),
        **{key: value["mean"] for key, value in s["metrics"].items()},
        "single_grasp_ms": s.get("latency", {}).get("single_grasp_ms"),
        "device": s.get("latency", {}).get("device"),
    })
pd.DataFrame(rows).sort_values(["run", "nfe"]).round(3)

,run,name,nfe,success_rate,collision_rate,contact_rate,single_grasp_ms,device
0,calibration,gt,NaN,0.989,0.000,1.000,NaN,None
1,calibration,random,NaN,0.001,0.221,0.056,NaN,None
2,equigraspflow_full,SE3_Euler-1,1.0,0.007,0.560,0.202,96.659,cuda:0
4,equigraspflow_full,SE3_Euler-2,2.0,0.111,0.634,0.519,97.600,cuda:0
6,equigraspflow_full,SE3_Euler-5,5.0,0.620,0.282,0.911,115.280,cuda:0
3,equigraspflow_full,SE3_Euler-10,10.0,0.937,0.048,0.998,147.280,cuda:0
5,equigraspflow_full,SE3_Euler-20,20.0,0.962,0.025,0.999,212.035,cuda:0
7,equigraspflow_full,SE3_RK_mk-20,80.0,0.964,0.025,1.000,578.606,cuda:0


## Save to Drive

In [13]:
!mkdir -p "{DRIVE_OUT}/results" "{DRIVE_OUT}/cache"
!rsync -a results/ "{DRIVE_OUT}/results/"
!rsync -a data/cache/ "{DRIVE_OUT}/cache/"
!ls "{DRIVE_OUT}/results/mujoco"

calibration  equigraspflow_full


In [14]:
from google.colab import runtime
runtime.unassign()
